In [24]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
from jax import random
import numpy as np
from lv import LV
from triangular_transport.flows.dataloaders import log_normal_reference_sampler,gaussian_reference_sampler
jax.config.update("jax_enable_x64", True)

In [34]:
seed = 0
no_samples = 1
sigma = float(jnp.sqrt(0.1))  # likelihood noise used inside LV.likelihood_pdf
u_true = jnp.array([0.83194674, 0.04134147, 1.0823151, 0.03991483])

lv = LV(
    seed=seed,
    no_samples=no_samples,
    prior_sampler=log_normal_reference_sampler,
    likelihood_sampler=log_normal_reference_sampler,
    normalize=False,
    u_true=u_true,
    sigma=sigma,
)

# ---- Construct a synthetic observation y_obs from the model using u_true ----
# y_obs should match what likelihood_pdf expects:
# likelihood_pdf flattens y and compares log(y) to xt.ravel().
ys = lv.solve_lv(lv.y0_start, u_true)      # shape (len(ts), 2)
xt = jnp.abs(ys).ravel()                   # "mean" in log-space inside likelihood_pdf
key = random.key(123)
y_obs = jnp.log(xt) + sigma * random.normal(key, shape=xt.shape)
yobs = np.load("y_obs.npy")

In [35]:
y_obs


Array([ 4.42785888,  3.25289221,  0.86993848,  3.4717829 ,  1.38219741,
        1.56881805,  2.67671261,  0.72577445,  4.50358609,  0.88268946,
        1.93081626,  4.55355253,  0.68726665,  2.67970075,  2.06663377,
        0.82322786,  3.18475386, -0.06028724], dtype=float64)

In [36]:
key2 = random.key(999)
u_near = u_true * jnp.exp(0.05 * random.normal(key2, shape=u_true.shape))
u_far  = u_true * jnp.exp(1.00 * random.normal(key2, shape=u_true.shape))

# ---- 1) internal consistency checks ----
post = lv.posterior(y_obs, u_near)
like = lv.likelihood_pdf(y_obs, u_near)
prior = lv.prior_pdf(u_near)

rel_err = jnp.abs(post - like * prior) / (jnp.abs(post) + 1e-12)
print("posterior vs like*prior rel err:", float(rel_err))

posterior vs like*prior rel err: nan


In [37]:
logpost = lv.log_posterior(y_obs, u_near)
loglike = lv.log_likelihood_pdf(y_obs, u_near)
logprior = lv.log_prior_pdf(u_near)
rel_err_log = jnp.abs(logpost - (loglike + logprior)) / (jnp.abs(logpost) + 1e-12)
print("log_posterior vs loglike+logprior rel err:", float(rel_err_log))

log_posterior vs loglike+logprior rel err: nan


In [9]:
logpost

Array(-4951.03937714, dtype=float64)

In [10]:
loglike

Array(-4955.05639585, dtype=float64)

In [24]:
x = jnp.squeeze(u_near)
y = jnp.ravel(y_obs)

# Lognormal requires y > 0
bad = jnp.any(y <= 0.0)

xt = lv.solve_lv(lv.y0_start, x)
xt = jnp.ravel(jnp.abs(xt))  # consider removing abs later

d = y.shape[0]
var = lv.sigma**2

resid = jnp.log(y) - xt
quad = jnp.sum(resid * resid) / var

# log(1/prod(y)) = -sum(log(y))
# Normal constant: -(d/2)log(2π var)
log_like = -jnp.sum(jnp.log(y)) - 0.5 * d * jnp.log(2 * jnp.pi * var) - 0.5 * quad

In [25]:
quad

Array(inf, dtype=float32)

In [26]:
xt

Array([97.99173   , 16.966204  ,  3.6686711 , 37.107506  ,  3.7121332 ,
        5.9379506 , 13.037373  ,  1.3223205 , 53.593376  ,  1.5874029 ,
       32.51718   , 74.54871   ,  2.6475344 , 17.419928  ,  5.934492  ,
        2.9488096 , 23.351818  ,  0.99813426], dtype=float32)

In [7]:
resid

Array([         inf,   9.395155  ,  -1.0339332 ,  -1.751297  ,
        -0.07836008,  -1.2455125 ,   3.187477  ,   0.3274747 ,
        24.453217  ,   1.5495309 , -25.622066  ,  -4.3838806 ,
        -0.47206354,  -6.795928  ,   3.1668339 ,  -1.7048544 ,
        20.84591   ,   0.7065799 ], dtype=float32)

In [36]:
a = LV(seed=1, no_samples = 100, prior_sampler=log_normal_reference_sampler, likelihood_sampler=log_normal_reference_sampler, sigma=jnp.sqrt(0.1).item())

In [37]:
x = jnp.ones(4)
a.prior_pdf(x)

Array(1.4956374e-09, dtype=float32)

In [38]:
y = jnp.ones(18)
a.likelihood_pdf(y, x)

Array(0., dtype=float32)

In [8]:
xt = a.solve_lv(jnp.array([30, 1]), x)
xt = xt.ravel()

In [40]:
y_dim = 18
# var_like = jnp.sqrt(0.1) ** 2
var_like = a.sigma ** 2
cov_mat = jnp.diag(jnp.full(y_dim, var_like))
inv_cov_mat = jnp.diag(jnp.full(y_dim, 1 / var_like))

In [41]:
jnp.exp((-1 / 2) * jnp.sum((jnp.log(y) - xt) * (inv_cov_mat @ (jnp.log(y) - xt))))

Array(0., dtype=float32)

In [42]:
(1 / (jnp.prod(y))) * (1 / (2 * jnp.pi)) ** (int(y_dim) / 2) * (1 / jnp.prod(jnp.diag(cov_mat))) ** (0.5)

Array(65.52109, dtype=float32)

In [34]:
1 / jnp.prod(jnp.diag(cov_mat)) ** (0.5)

Array(9.999998e+08, dtype=float32)

In [28]:
jnp.sqrt(0.1)

Array(0.31622776, dtype=float32, weak_type=True)

In [25]:
(1 / jnp.prod(jnp.diag(cov_mat))) ** (0.5)

Array(inf, dtype=float32)

In [21]:
jnp.exp((jnp.sum((jnp.log(y) - xt) * (inv_cov_mat @ (jnp.log(y) - xt)))) / -2)

Array(0., dtype=float32)

In [9]:
d = jnp.ones((9, 1))
t = jnp.zeros((9, 1))
dt = jnp.hstack([d, t])
dt.ravel()

Array([1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
       0.], dtype=float32)

In [8]:
len(jnp.arange(0 + 2, 20, step=2))

9

In [ ]:
a.prior_pdf(x)

Array(1.4956374e-09, dtype=float32)

: 

In [15]:
a.u_dim

4

In [16]:
(1 / (2 * jnp.pi)) ** (int(a.u_dim) / 2)

0.025330295910584447

In [19]:
cov_mat = jnp.diag(jnp.full(a.u_dim, a.std_prior))
(1 / jnp.prod(jnp.diag(cov_mat))) ** (0.5)

Array(2., dtype=float32)

In [26]:
inv_cov_mat = jnp.diag(jnp.full(a.u_dim, 1 / a.std_prior))
(-1 / 2) * (jnp.log(x) - a.mu_base) * (inv_cov_mat @ (jnp.log(x) - a.mu_base))

Array([-0.01104854, -6.3639607 , -0.01104854, -6.3639607 ], dtype=float32)

In [6]:
mu = jnp.array([-0.125, -3.0, -0.125, -3.0])
sigma = 1/jnp.sqrt(2)

x = jnp.ones(4)

Sigma_inv = jnp.eye(4) / sigma**2
det_Sigma = sigma**(2*4)

quad = (jnp.log(x)-mu) @ Sigma_inv @ (jnp.log(x)-mu)

pdf = (
    jnp.exp(-0.5*quad)
    / ((2*jnp.pi)**2 * jnp.sqrt(det_Sigma))
)

print(pdf)

1.4956374e-09


In [1]:
import numpy as np

In [22]:
true_us = np.load("true_us_obs.npy")

In [23]:
np.unique(true_us)

array([0.03991483, 0.04134147, 0.83194674, 1.0823151 ])